# Uber Fare Prediction (Linear Regression and Random Forest)

**Dataset:** `uber.csv`  
**Goal:** Predict `fare_amount` from trip distance, time and passenger count. Includes cleaning, outlier removal, correlation heatmap, and a comparison of two regression models.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

## 2. Load and clean

In [ ]:
import os

def find_file(name):
    """Look for a dataset in the usual places; in Google Colab, ask for an upload."""
    folders = [".", "data", "/content", "/content/drive/MyDrive", "/mnt/data",
               os.path.expanduser("~/Downloads"), os.path.expanduser("~/Desktop")]
    for folder in folders:
        path = os.path.join(folder, name)
        if os.path.exists(path):
            print(f"Found dataset: {path}")
            return path
    try:
        from google.colab import files   # only exists in Google Colab
        print(f"'{name}' not found. Please upload it now:")
        files.upload()
        if os.path.exists(name):
            return name
    except ImportError:
        pass
    raise FileNotFoundError(
        f"Could not find '{name}'. Put it in the same folder as this notebook and re-run."
    )

In [ ]:
df = pd.read_csv(find_file("uber.csv"))
print("Raw shape:", df.shape)

df = df.drop(columns=["Unnamed: 0", "key"], errors="ignore")
df = df.dropna()
print("After dropping missing values:", df.shape)
df.head()

## 3. Feature engineering
Distance from coordinates (haversine formula), plus hour and year from the pickup time.

In [ ]:
# distance in km
lat1 = np.radians(df["pickup_latitude"])
lat2 = np.radians(df["dropoff_latitude"])
dlat = lat2 - lat1
dlon = np.radians(df["dropoff_longitude"] - df["pickup_longitude"])
a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
df["distance"] = 6371 * 2 * np.arcsin(np.sqrt(a))

# hour and year from the pickup time
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], errors="coerce")
df = df.dropna(subset=["pickup_datetime"])
df["hour"] = df["pickup_datetime"].dt.hour
df["year"] = df["pickup_datetime"].dt.year
df = df.drop(columns=["pickup_datetime"])
df.head()

## 4. Remove wrong values and outliers

In [ ]:
# wrong values
df = df[(df["fare_amount"] > 0) & (df["distance"] > 0)]
df = df[(df["passenger_count"] > 0) & (df["passenger_count"] <= 6)]
print("After removing wrong values:", df.shape)

# outliers (IQR method)
for col in ["fare_amount", "distance"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    df = df[(df[col] >= q1 - 1.5 * iqr) & (df[col] <= q3 + 1.5 * iqr)]
print("After removing outliers:", df.shape)

## 5. Correlation

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap")
plt.show()

## 6. Train and test

In [ ]:
X = df.drop(columns=["fare_amount"])
y = df["fare_amount"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    print(name)
    print("R2   :", r2)
    print("RMSE :", rmse)
    print()
    return r2, rmse

## 7. Linear Regression

In [ ]:
lr_r2, lr_rmse = evaluate("Linear Regression", LinearRegression())

## 8. Random Forest

In [ ]:
rf_r2, rf_rmse = evaluate("Random Forest", RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1))

## 9. Comparison

In [ ]:
results = pd.DataFrame(
    {"R2": [lr_r2, rf_r2], "RMSE": [lr_rmse, rf_rmse]},
    index=["Linear Regression", "Random Forest"],
)
results